In [ ]:
import sys
from pathlib import Path

working_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (working_dir, *working_dir.parents) if (path / 'factor_gfn').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('无法从当前目录向上找到包含 factor_gfn/ 的项目根目录')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('项目根目录:', PROJECT_ROOT)


# Daily-Derived v1 N=1/2 Exact-TB 资源构建

人工运行入口。它只构建独立的 Daily-Derived exhaustive registry 和 exact TB logZ；不会创建 Stage 5 run，也不会启动训练。Raw registry 仅作为受保护路径常量，不会被打开。

In [ ]:
import json
from dataclasses import asdict

from factor_gfn.gfn import (
    ExhaustivePlanningConfig,
    ExhaustiveRegistry,
    ExpressionFeatureSpec,
    HybridVarianceTrainer,
    RealRewardDataConfig,
    RealRewardDataPaths,
    RealRewardProvider,
    build_real_reward_data_context,
    build_stage5_hybrid_variance_5_15_config,
    resolve_exhaustive_plan,
)
from factor_gfn.gfn.diagnostic_support import build_or_resume_n1_n2_registry
from factor_gfn.grammar import (
    DAILY_DERIVED_ACTION_REGISTRY,
    DAILY_DERIVED_V1_FEATURE_SPACE,
    SearchSpaceConfig,
)

RUN_REAL_EXHAUSTIVE = False
APPROVE_EXPLICIT_INCLUDE_OVER_BUDGET = True  # 2026-08-18 已人工批准 1712 candidates
RAW_REGISTRY_PATH = (
    PROJECT_ROOT / 'runs' / 'complexity_diagnostic_6_20'
    / 'manual_diagnostic_6_20_seed42' / 'exhaustive_registry.sqlite3'
)
OUTPUT_DIR = PROJECT_ROOT / 'runs' / 'daily_derived_v1' / 'exact_tb_n1_n2'
DERIVED_REGISTRY_PATH = OUTPUT_DIR / 'exhaustive_registry.sqlite3'

if DERIVED_REGISTRY_PATH.resolve() == RAW_REGISTRY_PATH.resolve():
    raise RuntimeError('Derived registry path 不得与 Raw registry 相同')
if DAILY_DERIVED_ACTION_REGISTRY.action_count != 152:
    raise RuntimeError('Daily-Derived action count 不再是冻结的 152')
print({
    'run_real_exhaustive': RUN_REAL_EXHAUSTIVE,
    'derived_registry_path': str(DERIVED_REGISTRY_PATH),
    'protected_raw_registry_path': str(RAW_REGISTRY_PATH),
    'action_space_fingerprint': DAILY_DERIVED_ACTION_REGISTRY.fingerprint(),
})


In [ ]:
planning = ExhaustivePlanningConfig(
    explicit_include_node_counts=(1, 2),
    approve_explicit_include_over_budget=APPROVE_EXPLICIT_INCLUDE_OVER_BUDGET,
)
plan = resolve_exhaustive_plan(
    SearchSpaceConfig(max_depth=2, max_nodes=2),
    planning,
    action_registry=DAILY_DERIVED_ACTION_REGISTRY,
)
counts = {
    node_count: plan.count_result(node_count).canonical_terminal_count
    for node_count in (1, 2)
}
if counts != {1: 16, 2: 1696}:
    raise RuntimeError(f'Derived canonical counts 漂移: {counts}')
if plan.resolved_exhaustive_node_counts != (1, 2):
    raise RuntimeError('Derived plan 未批准完整 N=1/2 exhaustive strata')
print(json.dumps({
    'counts': counts,
    'total_candidates': sum(counts.values()),
    'estimated_seconds': plan.resolved_estimated_evaluation_seconds,
    'over_budget_approval_used': plan.explicit_over_budget_approval_used,
    'plan_fingerprint': plan.fingerprint(),
    'vocabulary': plan.manifest()['vocabulary'],
}, ensure_ascii=False, indent=2))


In [ ]:
config = build_stage5_hybrid_variance_5_15_config(
    max_cycles=1,
    feature_space=DAILY_DERIVED_V1_FEATURE_SPACE,
)
data_config = RealRewardDataConfig()
data_paths = RealRewardDataPaths(
    expression_features=ExpressionFeatureSpec.daily_derived(),
)
required_paths = [
    data_paths.tensor_path,
    data_paths.expression_tensor_path,
    data_paths.expression_metadata_path,
    data_paths.universe_mask_path,
    data_paths.date_list_path,
    data_paths.stock_list_path,
    data_paths.processed_metadata_path,
    data_paths.industry_path,
    data_paths.industry_metadata_path,
    data_paths.barra_paths.metadata_path,
    data_paths.barra_paths.market_return_path,
    *[data_paths.barra_paths.exposure_path(name) for name in (
        'market_beta', 'size', 'momentum', 'volatility', 'liquidity'
    )],
]
missing = [str(path) for path in required_paths if not path.is_file()]
if missing:
    raise FileNotFoundError({'missing_preflight_paths': missing})

context = build_real_reward_data_context(data_config, data_paths)
provider = RealRewardProvider(context, config.reward)
if context.expression_feature_space_id != 'daily_derived_v1':
    raise RuntimeError('RealReward context 未使用 Daily-Derived expression tensor')
if context.manifest['label_formula'] != 'open[t+6] / open[t+1] - 1':
    raise RuntimeError('label contract 漂移')
print(json.dumps({
    'feature_space_id': context.expression_feature_space_id,
    'expression_shape': list(context.expression_feature_tensor.shape),
    'label_formula': context.manifest['label_formula'],
    'context_fingerprint': context.fingerprint,
    'provider_fingerprint': provider.fingerprint(),
    'validation_oos_loaded': provider.manifest()['validation_oos_loaded'],
}, ensure_ascii=False, indent=2))


## 人工执行闸门

确认上方 count、vocabulary、数据上下文和输出路径后，手工把 `RUN_REAL_EXHAUSTIVE` 改成 `True`，再执行下一格。中断后可从同一路径恢复。

In [ ]:
if not RUN_REAL_EXHAUSTIVE:
    raise RuntimeError('Safety stop: 未启动真实 exhaustive evaluation')
if not APPROVE_EXPLICIT_INCLUDE_OVER_BUDGET:
    raise RuntimeError('N=1/2 超预算批准未启用')

registry = build_or_resume_n1_n2_registry(
    DERIVED_REGISTRY_PATH,
    provider,
    reward_floor=config.reward.reward_floor,
    action_registry=DAILY_DERIVED_ACTION_REGISTRY,
    approve_explicit_include_over_budget=True,
    progress_every=25,
)
registry.close()
print('Derived exhaustive evaluation 与 exact mass 聚合已完成:', DERIVED_REGISTRY_PATH)


In [ ]:
read_only_registry = ExhaustiveRegistry(
    DERIVED_REGISTRY_PATH,
    read_only=True,
    action_registry=DAILY_DERIVED_ACTION_REGISTRY,
)
try:
    trainer = HybridVarianceTrainer(config, provider, device='cpu')
    semantics = trainer.target_exhaustive_reuse_semantics()
    proofs = trainer.configure_hybrid_exhaustive_registry(
        read_only_registry,
        source_semantics_by_N={1: semantics, 2: semantics},
    )
    verification = {
        node_count: {
            'coverage': read_only_registry.coverage(node_count),
            'exact_mass': asdict(read_only_registry.exact_mass_result(node_count)),
            'reuse_proof_fingerprint': proofs[node_count].proof_fingerprint,
        }
        for node_count in (1, 2)
    }
    print(json.dumps(verification, ensure_ascii=False, indent=2))
finally:
    read_only_registry.close()
print({'DERIVED_EXACT_TB_READY': True, 'STAGE5_TRAINING_EXECUTED': False})
